# ANN recall vs cost

Trade off recall@k against memory and latency for an HNSW index.
Hardcoded coefficients are rules of thumb from public benchmarks
(ann-benchmarks.com, Pinecone / Qdrant blogs, 2023-2025) and your numbers
will differ; this is for **shape**, not point estimates.

In [ ]:
def hnsw_estimate(n_vectors, dim, M=32, efSearch=200, bytes_per_float=4):
    # raw vector storage
    raw_gb = n_vectors * dim * bytes_per_float / 1e9
    # graph overhead: each node stores ~M neighbours, 4 bytes each, plus the upper levels (~M extra)
    graph_gb = n_vectors * (M * 2) * 4 / 1e9
    total_gb = raw_gb + graph_gb
    # rough query cost: O(efSearch * log(N)) distance comps
    import math
    dist_comps_per_q = efSearch * max(1, math.log2(n_vectors))
    # ~1 GFLOP per 1M dim-128 distance comps on modern CPU (vector instructions)
    flops_per_q = dist_comps_per_q * dim * 2
    return {
        'raw_gb':           round(raw_gb, 2),
        'graph_gb':         round(graph_gb, 2),
        'total_gb':         round(total_gb, 2),
        'dist_comps_per_q': int(dist_comps_per_q),
        'flops_per_q':      int(flops_per_q),
    }

for n in [1_000_000, 10_000_000, 100_000_000, 1_000_000_000]:
    for dim in [384, 768, 1536]:
        r = hnsw_estimate(n, dim)
        print(f'N={n:>13,}  dim={dim:<5d}  RAM={r["total_gb"]:>7.1f} GB  '
              f'dist/q={r["dist_comps_per_q"]:>7d}  FLOPs/q={r["flops_per_q"]:>12,d}')

## Switch to IVF-PQ

At ~100M vectors, HNSW RAM exceeds a single big-mem node. Move to IVF-PQ
(product quantization) to compress vectors 8-32x and accept a 1-3 point recall hit.
Or DiskANN if you want graph quality with disk residency (Subramanya et al., NeurIPS 2019).

In [ ]:
def ivf_pq_estimate(n_vectors, dim, pq_bytes=16, nlist=4096, nprobe=64):
    # PQ: each vector stored as pq_bytes bytes
    pq_gb = n_vectors * pq_bytes / 1e9
    # centroids in RAM
    centroids_gb = nlist * dim * 4 / 1e9
    # query: scan nprobe lists, each ~ n/nlist vectors, distance comp on pq codes
    dist_comps_per_q = nprobe * (n_vectors / nlist)
    return {
        'pq_gb':            round(pq_gb, 2),
        'centroids_gb':     round(centroids_gb, 3),
        'total_gb':         round(pq_gb + centroids_gb, 2),
        'dist_comps_per_q': int(dist_comps_per_q),
        'compression_x':    round(dim*4/pq_bytes, 1),
    }

for n in [100_000_000, 1_000_000_000, 10_000_000_000]:
    r = ivf_pq_estimate(n, dim=768)
    print(f'N={n:>13,}  total RAM={r["total_gb"]:>7.1f} GB  '
          f'dist/q={r["dist_comps_per_q"]:>9d}  compression={r["compression_x"]}x')

Cost guidance:

- HNSW costs scale with **vectors x dim**; choose the smallest dim that holds recall.
- IVF-PQ costs scale with **vectors x pq_bytes**; pq_bytes is the main lever.
- The cheapest recall lift is a **cross-encoder reranker** over the top 100. See `05-vector-dbs-and-retrieval/`.